# 10 — Environment Design & Validation

A silent RL bug is worse than a loud one. If your environment has a reward calculation error, a leaky observation, or action clipping without communication, your agent will train to optimality on the wrong problem — and you may not discover this until deployment.

## Common environment bugs

**Reward normalisation inconsistency**: reward is clipped during training but not at eval — policy is optimal for the clipped version, not the true objective.

**Observation leakage**: the observation contains information the agent shouldn't have (opponent's hidden cards, future events). The agent trains perfectly by cheating.

**Action clipping without feedback**: the environment silently clips actions but doesn't tell the policy, breaking the policy gradient signal.

**Episode boundary bugs**: done=True is set one step too early or late, causing wrong terminal bootstrap.

**Reward sign flip**: a sign error causes the agent to maximise what should be minimised.

**Observation normalisation drift**: if you normalise observations with a running mean/std, the normaliser must be consistent between train and eval — drifting normalisation degrades deployed policy performance.

In [ ]:
# gymcheck: environment validation prototype

import numpy as np
import gymnasium as gym
from dataclasses import dataclass
from typing import Optional

@dataclass
class CheckResult:
    passed: bool
    check_name: str
    detail: str
    severity: str = "WARNING"

class GymCheck:
    def __init__(self, env_factory, n_steps=500):
        self.env_factory = env_factory
        self.n_steps = n_steps
        self.results = []

    def _check(self, passed, name, detail, severity="WARNING"):
        r = CheckResult(passed, name, detail, severity)
        self.results.append(r)
        return r

    def check_spaces(self, env):
        if hasattr(env.observation_space, 'high'):
            has_inf = np.any(np.isinf(env.observation_space.high)) or np.any(np.isinf(env.observation_space.low))
            self._check(not has_inf, "obs_space_bounded",
                "Obs space has infinite bounds -- normalisation required" if has_inf else "Obs space bounded",
                "WARNING" if has_inf else "INFO")

    def check_reset_consistency(self, env, n=5):
        shapes = set()
        for _ in range(n):
            obs, _ = env.reset()
            shapes.add(np.array(obs).shape)
        self._check(len(shapes)==1, "reset_shape_consistent",
                    f"Consistent shape {shapes}" if len(shapes)==1 else f"Inconsistent shapes: {shapes}",
                    "ERROR" if len(shapes)>1 else "INFO")

    def check_reward_range(self, env):
        rewards = []
        obs, _ = env.reset()
        for _ in range(self.n_steps):
            action = env.action_space.sample()
            obs, r, term, trunc, _ = env.step(action)
            rewards.append(r)
            if term or trunc: obs, _ = env.reset()
        rewards = np.array(rewards)
        all_zero = np.all(rewards == 0)
        self._check(not all_zero, "reward_not_all_zero",
                    "All rewards are zero -- possible reward bug" if all_zero else f"Reward range: [{rewards.min():.3f}, {rewards.max():.3f}]",
                    "ERROR" if all_zero else "INFO")
        if len(rewards) > 10:
            std = rewards.std()
            max_z = np.max(np.abs((rewards-rewards.mean())/(std+1e-8)))
            self._check(max_z < 10, "reward_no_spikes",
                        f"Max z-score {max_z:.1f} -- possible spike" if max_z>=10 else f"Reward distribution OK",
                        "WARNING" if max_z>=10 else "INFO")

    def check_determinism(self, env, seed=42):
        trajectories = []
        for trial in range(2):
            obs, _ = env.reset(seed=seed)
            traj = [np.array(obs).copy()]
            for _ in range(20):
                np.random.seed(seed + trial*1000)
                obs, _, term, trunc, _ = env.step(env.action_space.sample())
                traj.append(np.array(obs).copy())
                if term or trunc: break
            trajectories.append(traj)
        if len(trajectories[0]) == len(trajectories[1]):
            diffs = [np.max(np.abs(a-b)) for a,b in zip(trajectories[0], trajectories[1])]
            max_diff = max(diffs)
            self._check(max_diff < 1e-6, "determinism",
                        f"Non-deterministic (max diff={max_diff:.2e})" if max_diff>=1e-6 else "Deterministic given same seed",
                        "WARNING" if max_diff>=1e-6 else "INFO")

    def check_done_signal(self, env):
        obs, _ = env.reset()
        done = False
        for _ in range(self.n_steps):
            _, _, term, trunc, _ = env.step(env.action_space.sample())
            if term or trunc: done = True; break
        self._check(done, "episode_terminates",
                    "Episode terminated within step limit" if done else f"Never terminated in {self.n_steps} steps",
                    "WARNING" if not done else "INFO")

    def run(self):
        env = self.env_factory()
        print(f"gymcheck: {type(env).__name__}")
        print("="*50)
        self.check_spaces(env)
        self.check_reset_consistency(env)
        self.check_reward_range(env)
        self.check_determinism(env)
        self.check_done_signal(env)
        env.close()
        errors = [r for r in self.results if not r.passed and r.severity=="ERROR"]
        warnings = [r for r in self.results if not r.passed and r.severity=="WARNING"]
        for r in self.results:
            icon = "✓" if r.passed else ("!! ERROR" if r.severity=="ERROR" else "WARN")
            print(f"  {icon} [{r.check_name}]: {r.detail}")
        print(f"\nSummary: {len([r for r in self.results if r.passed])} passed | {len(warnings)} warnings | {len(errors)} errors")
        return self.results

for env_id in ["CartPole-v1", "LunarLander-v2"]:
    checker = GymCheck(lambda eid=env_id: gym.make(eid), n_steps=500)
    checker.run()
    print()


## Before every training run

```python
results = GymCheck(lambda: MyEnv()).run()
assert all(r.severity != 'ERROR' for r in results if not r.passed)
```

Also check manually:
- [ ] Render output matches expected states
- [ ] Action space dtype matches policy output (float vs int)
- [ ] Observation contains all necessary info and nothing it shouldn't
- [ ] Reward is on a learnable scale (approx [-1, 1])